<a href="https://colab.research.google.com/github/elahesadeghian/recommender-systems/blob/main/notebooks/01_movielens_item_item_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MovieLens Recommender — Item–Item KNN

This notebook builds a simple **item–item collaborative filtering** recommender on the MovieLens (small) dataset using cosine similarity.

**Goal:** Given a movie title, return Top-N similar movies.  
**Extras:** A simple 80/20 split with **Precision@10** evaluation (baseline).


In [15]:
# 1) Setup & Imports
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)

# If running on Colab, fetch the dataset (safe to re-run)
!wget -q http://files.grouplens.org/datasets/movielens/ml-latest-small.zip -O ml-latest-small.zip
!unzip -o -q ml-latest-small.zip


## 2) Load Dataset
We use **MovieLens small**:

- `ratings.csv` → (userId, movieId, rating, timestamp)  
- `movies.csv`  → (movieId, title, genres)

First, load and peek.

In [16]:
ratings = pd.read_csv("ml-latest-small/ratings.csv")
movies  = pd.read_csv("ml-latest-small/movies.csv")

print("Ratings shape:", ratings.shape, "| Movies shape:", movies.shape)
display(ratings.head(), movies.head())


Ratings shape: (100836, 4) | Movies shape: (9742, 3)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## 3) Build User–Item Matrix
Create a **movie × user** rating matrix `R`.  
We fill missing values with 0 (baseline) — good enough for a demo, not ideal for production.


In [17]:
R = ratings.pivot_table(index="movieId", columns="userId", values="rating").fillna(0)
R.shape


(9724, 610)

## 4) Item–Item Cosine Similarity
Compute cosine similarity between **movie vectors** (rows of `R`).  
This yields a square similarity matrix `sim_df` indexed by `movieId`.


In [18]:
sim = cosine_similarity(R)
sim_df = pd.DataFrame(sim, index=R.index, columns=R.index)
sim_df.shape


(9724, 9724)

## 5) Recommender Function
`recommend_by_title(title_query, topk)`:

1) find the first matching `movieId` for the query,  
2) sort most similar movies (exclude the movie itself),  
3) return **Top-K** titles with similarity scores.


In [19]:
def recommend_by_title(title_query: str, topk: int = 10) -> pd.DataFrame:
    """
    Return top-k movies most similar to the given title using item–item cosine similarity.

    Args:
        title_query: Partial or full movie title (case-insensitive).
        topk: Number of recommendations to return.

    Returns:
        DataFrame with columns [title, similarity], sorted desc. Empty if no title is found.
    """
    # find matching movieId
    match = movies[movies["title"].str.contains(title_query, case=False, na=False)]
    if match.empty:
        return pd.DataFrame(columns=["title", "similarity"])
    mid = int(match["movieId"].iloc[0])

    # sort similarities and drop the item itself
    scores = sim_df[mid].sort_values(ascending=False).iloc[1 : topk + 1]
    out = (scores.round(3).rename("similarity").reset_index()  # [movieId, similarity]
           .merge(movies[["movieId","title"]], on="movieId", how="left"))
    return out[["title", "similarity"]]


## 6) Example
Query with a known title and get Top-10 similar movies.


In [20]:
recommend_by_title("Toy Story", 10)


,title,similarity
0,Toy Story 2 (1999),0.573
1,Jurassic Park (1993),0.566
2,Independence Day (a.k.a. ID4) (1996),0.564
3,Star Wars: Episode IV - A New Hope (1977),0.557
4,Forrest Gump (1994),0.547
5,"Lion King, The (1994)",0.541
6,Star Wars: Episode VI - Return of the Jedi (1983),0.541
7,Mission: Impossible (1996),0.539
8,Groundhog Day (1993),0.534
9,Back to the Future (1985),0.530


## 7) Why cosine? Limits (sparsity & cold-start)

- **Cosine** measures the angle between item vectors; it’s scale-invariant and fast on sparse matrices.
- **Sparsity** → popularity bias (popular items look similar to many).
- **Cold-start** → new users/items with no ratings can’t be handled by pure CF.
- **Next** → mitigate sparsity via **matrix factorization (SVD)** or a **hybrid** with content features (genres).


## 8) Baseline Evaluation: 80/20 split + Precision@10
We split ratings randomly into **train (80%)** and **test (20%)**.

- Build item–item similarity on **train** only.  
- For each user, recommend Top-K based on mean similarity to items they **liked** in train (`rating ≥ 4`).  
- Compute **Precision@10** on test (fraction of recommended items that the user rated ≥ 4 in test).


In [21]:
# Shuffle & split
ratings_all = ratings.sample(frac=1, random_state=42)
cut = int(0.8 * len(ratings_all))
train = ratings_all.iloc[:cut].copy()
test  = ratings_all.iloc[cut:].copy()

# Rebuild R and similarity on TRAIN
R_train = train.pivot_table(index="movieId", columns="userId", values="rating").fillna(0)
sim_train = cosine_similarity(R_train)
sim_train_df = pd.DataFrame(sim_train, index=R_train.index, columns=R_train.index)

# Items each user liked in TRAIN (rating >= 4)
user_pos_items = (train[train["rating"] >= 4.0]
                  .groupby("userId")["movieId"].apply(set)
                  .to_dict())
all_items = set(R_train.index.tolist())

def topn_for_user(u: int, k: int = 10):
    liked = user_pos_items.get(u, set())
    if not liked:
        return []
    candidates = list(all_items - liked)
    sims = sim_train_df.loc[candidates, list(liked)].mean(axis=1)
    return list(sims.sort_values(ascending=False).head(k).index)

def precision_at_k_test(k: int = 10, threshold: float = 4.0) -> float:
    test_pos = (test[test["rating"] >= threshold]
                .groupby("userId")["movieId"].apply(set)
                .to_dict())
    precisions = []
    for u in set(test["userId"].unique()):
        recs = topn_for_user(u, k)
        if not recs:
            continue
        gt = test_pos.get(u, set())
        if not gt:
            continue
        hit = sum(1 for m in recs if m in gt)
        precisions.append(hit / k)
    return float(np.mean(precisions)) if precisions else float("nan")

prec_at10 = precision_at_k_test(k=10)
prec_at10


0.09529411764705883

## 9) Notes & Next Steps

- **Baseline**: This notebook implemented a simple item–item collaborative filtering with cosine similarity.  
- **Reported metric**: Precision@10 on a random 80/20 split (quick baseline evaluation).  
- **Limitations**: No hyperparameter tuning, popularity bias, and no handling of cold-start users/items.  
- **Next steps**:
  - Implement **Matrix Factorization (SVD)** using the `surprise` library.  
  - Evaluate with **RMSE** (5-fold CV) and compare against the item–item baseline.  
  - (Optional) Build a simple **Streamlit demo** for interactive recommendations.  
